# S01 toy — the agent loop, transparent

A tool-calling loop against a **mock model** (a plain Python function). No network, no keys, no cost. You can read the entire "model" and predict exactly what it will do — the loop's behavior becomes fully visible when the model is not a black box.

**How to use:** run cells in order. For each experiment, write your prediction as a comment *before* running. The gap between prediction and result is the lesson.

## The API shape

One request in, one response out. The API is **stateless** — every request carries the entire `messages` list. Two response kinds: a final answer (`content`, no `tool_calls`) or a tool request (`tool_calls`). The helpers below build both, API-shaped.

In [ ]:
import itertools
import json

_ids = itertools.count(1)

def _reply(text):
    msg = {"role": "assistant", "content": text, "tool_calls": []}
    return {"choices": [{"message": msg}], "usage": {"total_tokens": 10}}

def _tool_call(name, args):
    call = {"id": f"call_{next(_ids)}", "type": "function",
            "function": {"name": name, "arguments": json.dumps(args)}}
    msg = {"role": "assistant", "content": None, "tool_calls": [call]}
    return {"choices": [{"message": msg}], "usage": {"total_tokens": 15}}

def _validate(messages):
    """Like a real API, it rejects an orphaned tool result — that one failure class,
    not full protocol validation: the id must anchor to a prior assistant tool_call."""
    seen = set()
    for m in messages:
        if m["role"] == "assistant":
            for c in (m.get("tool_calls") or []):
                seen.add(c["id"])
        elif m["role"] == "tool" and m["tool_call_id"] not in seen:
            raise ValueError(
                f"orphaned tool result {m['tool_call_id']}: "
                "the assistant message carrying it was dropped or mutated")


In [ ]:
def mock_model(messages):
    """Rule-based stand-in for POST /v1/chat/completions. Fully inspectable."""
    _validate(messages)
    last = messages[-1]
    if last["role"] == "tool":
        return _reply(f"Based on the tool: {last['content']}")
    text = last["content"].lower()
    if "fragile" in text:
        return _tool_call("get_weather_fragile", {"city": "Madrid"})
    if "weather" in text:
        city = "Madrid" if "madrid" in text else "Nowhere"
        return _tool_call("get_weather", {"city": city})
    return _reply("Ask me about the weather (mention 'weather').")

def get_weather(city):
    return {"Madrid": "22C, sunny"}.get(city, "no data")

def get_weather_fragile(city):
    raise ConnectionError("weather service unreachable")

TOOLS = {"get_weather": get_weather, "get_weather_fragile": get_weather_fragile}

def dispatch(call):
    """Execute one tool_call; errors become tool results, not crashes."""
    try:
        return TOOLS[call["function"]["name"]](**json.loads(call["function"]["arguments"]))
    except Exception as exc:
        return f"ERROR: {type(exc).__name__}: {exc}"


## The loop

The whole agent: call the model → append the assistant message **verbatim** → if no `tool_calls`, done → else execute each call, append the results, repeat. Stop conditions: no `tool_calls` (success), the turn cap (bounded execution), or an error.

In [ ]:
def run_loop(user_prompt, model=mock_model, max_turns=6):
    messages = [{"role": "user", "content": user_prompt}]
    for turn in range(1, max_turns + 1):
        body = model(messages)
        msg = body["choices"][0]["message"]
        messages.append(msg)            # the line experiment 2 removes
        calls = msg.get("tool_calls") or []
        if not calls:
            return msg["content"], messages, turn
        for call in calls:
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": dispatch(call)})
    return None, messages, max_turns    # the turn cap is a harness property, not a model property


## Experiment 1 — happy path

**Predict first:** how many turns? What does each transcript row contain? Then run.

In [ ]:
answer, transcript, turns = run_loop("What's the weather in Madrid?")
for m in transcript:
    payload = m.get("content") if m.get("content") is not None else m.get("tool_calls")
    print(f"{m['role']:>9} -> {payload}")
print(f"\nFinal answer after {turns} turn(s): {answer}")


## Experiment 2 — drop the assistant message

**Predict first:** what fails, and where? Then run. (The mock reproduces the orphaned-tool-result failure class the way a real API does —
one check, not full protocol validation.)

In [ ]:
def run_loop_broken(user_prompt, max_turns=3):
    messages = [{"role": "user", "content": user_prompt}]
    for turn in range(1, max_turns + 1):
        body = mock_model(messages)
        msg = body["choices"][0]["message"]
        # BUG: messages.append(msg) is missing — the assistant turn vanishes
        calls = msg.get("tool_calls") or []
        if not calls:
            return msg["content"], messages, turn
        for call in calls:
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": dispatch(call)})
    return None, messages, max_turns

try:
    run_loop_broken("What's the weather in Madrid?")
except ValueError as exc:
    print(f"The API rejected the next request:\n  {exc}")


## Experiment 3 — the model that never stops

**Predict first:** what ends the loop, and after how many turns?

In [ ]:
def mock_model_forever(messages):
    _validate(messages)
    return _tool_call("get_weather", {"city": "Madrid"})   # never stops calling tools

answer, transcript, turns = run_loop("weather?", model=mock_model_forever, max_turns=4)
print(f"answer={answer!r}  turns={turns}")
print("The model never stopped — the turn cap (a harness property) did.")


## Experiment 4 — the tool that raises

**Predict first:** does the loop crash, or does the error become data? Where does it surface?

In [ ]:
answer, transcript, turns = run_loop("Is the fragile weather service up in Madrid?")
for m in transcript:
    payload = m.get("content") if m.get("content") is not None else "[tool_call]"
    print(f"{m['role']:>9} -> {payload}")


## What transfers to your real harness

- `mock_model` → your real model call over HTTP (auth, a long timeout, token accounting).
- `dispatch` → your tool executor's try/except: errors as tool results, never crashes.
- the append-verbatim line → why real APIs reject orphaned tool results.
- `max_turns` → a harness property chosen from evidence, not intuition — the kind of call your decision log records.
- what the toy doesn't have: a real workspace, a path jail, shell access — where containment stops being a toy question.

Now do the real build in your own project. You type it.